# core

> Kosha is a tool for building a context for code generation based on your repo and environment.
> It uses a vector database to store code snippets and their embeddings, allowing you to search for relevant code based
> on natural language queries.
> The core functionality includes managing the database, updating package metadata, embedding code snippets,
> and performing context-aware searches.

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastprogress/fastprogress.py:171: UserWarning: Couldn't import ipython display functions, progress bar will use console behavior
  warn("Couldn't import ipython display functions, progress bar will use console behavior")


In [ ]:
#| export
import ast, os, re
from ast import get_source_segment as gs
from collections import defaultdict, Counter
from apswutils.utils import hash_record
from importlib.metadata import version as v, metadata as meta, distribution as dist
from chonkie import AutoEmbeddings
from fastcore.all import (Path, first, patch, timed_cache, L, parallel, merge, AttrDict,
                          filter_keys, not_, in_, listify, true, fdelegates, parallel_async,type2str)
from fastcore.xdg import xdg_data_home
from tqdm import tqdm
from json import loads as jl
from litesearch import *

## Utils
> Utility functions for finding the repo root, copying the SKILL.md file, running async code from sync contexts,
> and getting installed package versions.

In [ ]:
#| export
@timed_cache(maxsize=512)
def parse(code=None, p=None):
    "Parse source, tag parents, return (tree, imp, top_fns, all_fns). Cached — called freely."
    if not code and not p: return None, {}, set(), set()
    try: tree = ast.parse(Path(p).read_text(errors='replace') if not code else code)
    except SyntaxError: return None, {}, set(), set()
    [setattr(c,'parent',n) for n in ast.walk(tree) for c in ast.iter_child_nodes(n)]
    is_top = lambda n: isinstance(getattr(n,'parent',None), ast.Module)
    is_fn  = lambda n: isinstance(n,(ast.FunctionDef,ast.AsyncFunctionDef,ast.ClassDef))
    is_nm  = lambda n: isinstance(n, ast.Name)
    imp, imp2mod = {}, {}
    for n in ast.walk(tree):
        if isinstance(n, ast.Import):
            for a in n.names: imp[a.asname or a.name.split('.')[0]] = a.name.split('.')[0]
        elif isinstance(n, ast.ImportFrom):
            pkg = (n.module or '').split('.')[0]
            for a in n.names:
                if a.name != '*':
	                imp[a.asname or a.name] = pkg
	                if n.module: imp2mod[a.asname or a.name] = f'{n.module}.{a.name}'
                else: imp.setdefault(f'*{pkg}', pkg)
    top = {n.name for n in ast.walk(tree) if is_fn(n) and is_top(n)}
    ca  = {nm for n in tree.body if isinstance(n,ast.Assign) and isinstance(n.value,ast.Call)
           for t in n.targets
           for nm in ([t.id] if is_nm(t) else [e.id for e in t.elts if is_nm(e)] if isinstance(t,ast.Tuple) else [])}
    return tree, imp, top, top | ca, imp2mod

In [ ]:
#| export
def repo_root() -> Path:
	'Find the root of the current git repository, or None if not in a repo.'
	return first((Path.cwd(), *Path.cwd().parents), lambda p: (p/'.git').exists())

def mv_skill_md(dry_run=True, dir=None) -> None:
	"Copy bundled SKILL.md to `.agents/skills/kosha/` at project root or specified dir."
	if not (r := dir or repo_root()): return
	base = Path(__file__).parent if '__file__' in globals() else Path.cwd()
	if not (src := base.joinpath('SKILL.md')).exists(): return
	dest = Path(r).joinpath('.agents', 'skills', 'kosha', 'SKILL.md')
	if dry_run: print(f'Would copy {src} to {dest}')
	else: dest.mk_write(src.read_text(encoding='utf-8'))

In [ ]:
#| export
def arun(coro) -> any:
	'Run an async coroutine from sync code, even if already inside an event loop'
	import asyncio
	try: asyncio.get_running_loop()
	except RuntimeError: return L(asyncio.run(coro))
	import concurrent.futures
	with concurrent.futures.ThreadPoolExecutor() as pool: return L(pool.submit(asyncio.run, coro).result())

In [ ]:
async def _add(a, b): return a + b
assert arun(_add(1, 2)) == L(3)
print('arun: sync context ok')

arun: sync context ok


In [ ]:
async def _test_nested():
	result = arun(_add(10, 20))
	assert first(result) == 30, f'got {result}'
arun(_test_nested())
print('arun: nested-loop context ok')

arun: nested-loop context ok


In [ ]:
mv_skill_md()

Would copy /Users/71293/code/personal/orgs/kosha/nbs/SKILL.md to /Users/71293/code/personal/orgs/kosha/.agents/skills/kosha/SKILL.md


In [ ]:
mv_skill_md(dir='.')

Would copy /Users/71293/code/personal/orgs/kosha/nbs/SKILL.md to .agents/skills/kosha/SKILL.md


In [ ]:
#| export
_req_nm = re.compile(r'^[\w][\w.-]*')

def pkg_trans_deps(seeds:list, depth:int=2) -> L:
	'BFS over importlib.metadata requires: return seeds + all transitive deps up to depth levels (installed only, no optional extras).'
	seen, frontier = set(seeds), set(seeds)
	for _ in range(depth):
		nxt = set()
		for pkg in frontier:
			try:
				for req in (dist(pkg).requires or []):
					if 'extra ==' in req: continue
					if (m := _req_nm.match(req)) and spec(m.group(0)): nxt.add(m.group(0))
			except Exception: pass
		nxt -= seen
		if not nxt: break
		seen |= nxt; frontier = nxt
	return L(seen)

In [ ]:
#| export
def env_pkg_versions(pyproject=True, depth:int=1) -> dict:
	'''Get a dict of installed package versions using importlib.metadata.
	passing depth traverse multiple layers of dependencies'''
	pkgs = installed_packages(pyproject=pyproject)
	if pyproject and depth: pkgs = pkg_trans_deps(pkgs, depth)
	return merge(*pkgs.map(lambda p: {dist(p).metadata['Name']: dist(p).version}))

In [ ]:
installed_packages(pyproject=True)

['fastprogress', 'litesearch', 'pyan3', 'watchfiles']

In [ ]:
pkg_trans_deps(['litesearch'], depth=1)

['chonkie', 'pdf-oxide', 'onnxruntime', 'notebook', 'codesigs', 'litesearch', 'model2vec', 'yake', 'pandas', 'fastlite', 'tokenizers', 'pillow', 'usearch']

In [ ]:
# httpx depends on httpcore; depth=1 should include both httpx + httpcore
d1 = pkg_trans_deps(['httpx'], depth=1)
assert 'httpx' in d1, "seeds must be in result"
assert 'httpcore' in d1, f"depth=1 should include httpcore (direct dep of httpx): {d1}"
# depth=0 returns only seeds
assert list(pkg_trans_deps(['httpx'], depth=0)) == ['httpx'], "depth=0 should return only seeds"
# pyproject seeds → depth=2 expansion
pyp = installed_packages(pyproject=True)
if pyp:
    deep = pkg_trans_deps(list(pyp), depth=2)
    assert set(pyp).issubset(set(deep)), "seeds must be a subset of result"
    assert len(deep) >= len(pyp), "depth=2 should expand beyond pyproject direct deps"
    print(f"pkg_trans_deps ok: {len(pyp)} pyproject seeds → {len(deep)} packages at depth=2")
else:
    print("pkg_trans_deps ok (no pyproject deps to test)")

pkg_trans_deps ok: 4 pyproject seeds → 57 packages at depth=2


In [ ]:
env_pkg_versions()

{'Jinja2': '3.1.6',
 'model2vec': '0.8.1',
 'onnxruntime': '1.25.1',
 'pillow': '12.2.0',
 'pandas': '3.0.2',
 'watchfiles': '1.1.1',
 'usearch': '2.25.1',
 'chonkie': '1.6.4',
 'notebook': '7.5.5',
 'pyan3': '2.5.0',
 'codesigs': '0.0.2',
 'litesearch': '0.0.24',
 'anyio': '4.13.0',
 'yake': '0.7.3',
 'fastcore': '1.12.43',
 'tokenizers': '0.23.1',
 'fastprogress': '1.1.5',
 'fastlite': '0.2.4',
 'python-fasthtml': '0.13.4',
 'pdf_oxide': '0.3.39'}

## embedder
> jina-embeddings-v2-base-code is a model designed for code understanding and generation tasks

In [ ]:
#| export
cr_instr = AttrDict(query="Represent this query for searching relevant code: {text}", document="{text}")
model = AttrDict(model='mrsladoje/CodeRankEmbed-onnx-int8', onnx_path='onnx/model.onnx', prompt=cr_instr)
strict_skip_folder_re = r'^[._]|^(?:tests?|examples?|docs?|build|dist)$'
strict_skip_file_re = r'^[._]|^(?:setup\.py|conftest\.py)$'

@timed_cache(3600*24)
def embedder(
    emb_model=model  # model config AttrDict (model name, onnx_path, prompt instructions)
) -> 'FastEncode':
    "Load or return cached CodeRankEmbed ONNX document embedder (24h TTL)."
    return FastEncode(emb_model, parallel=8, batch_size=64)

@timed_cache(3600*24)
def static_embedder(
    emb_model='minishlab/potion-retrieval-32M'  # HuggingFace model id for static embeddings
) -> 'AutoEmbeddings':
    "Load or return cached static (potion-retrieval) embedder (24h TTL)."
    return AutoEmbeddings().get_embeddings(emb_model)

def emb_doc(
    e  # embedder factory — callable returning a FastEncode or AutoEmbeddings instance
):
    "Curry an embedder factory into a document-embedding function `t -> embedding`."
    return lambda t: e().encode_document(t) if isinstance(e(), FastEncode) else e().embed_batch(listify(t))

def emb_query(
    e  # embedder factory — callable returning a FastEncode or AutoEmbeddings instance
):
    "Curry an embedder factory into a query-embedding function `t -> embedding`."
    return lambda t: e().encode_query(t) if isinstance(e(), FastEncode) else e().embed_batch(listify(t))

In [ ]:
emb_doc(embedder)('def add(a, b): return a + b')

array([[ 1.1848e-02, -3.0869e-02,  2.4231e-02, -1.4404e-02, -8.3590e-04,
         4.9469e-02, -4.8828e-02, -2.7969e-02, -9.0027e-03, -2.9724e-02,
         4.6539e-02,  6.9458e-02, -1.0262e-02, -4.5319e-02,  1.8244e-03,
        -4.2694e-02,  3.5950e-02, -1.4320e-02, -5.1086e-02,  5.8098e-03,
         1.0651e-01, -4.6417e-02,  1.8097e-02, -4.7485e-02,  5.7495e-02,
        -3.5004e-02, -5.4047e-02,  1.2405e-02,  8.4152e-03,  3.8361e-02,
        -7.0129e-02,  6.0272e-02,  3.6346e-02,  2.6718e-02,  2.9480e-02,
         2.7222e-02,  3.1738e-02, -3.4058e-02, -5.7465e-02, -3.3966e-02,
        -2.2888e-02,  5.6030e-02,  3.2043e-02,  2.3209e-02,  4.4769e-02,
         6.5651e-03, -3.4142e-03,  1.6022e-02,  7.7515e-02, -2.0081e-02,
         3.2883e-03, -7.1045e-02, -7.4341e-02,  7.9575e-03,  2.4155e-02,
         1.9791e-02, -5.1880e-03, -2.0493e-02,  5.5237e-02, -2.2171e-02,
         7.5989e-02,  7.3181e-02,  4.8752e-03,  2.5543e-02,  8.8730e-03,
        -2.0798e-02,  9.5825e-03,  3.5309e-02,  1.0

## Kosha
> The main class for managing the code and environment databases, updating package metadata, embedding code snippets,
> and performing context-aware searches.

In [ ]:
#| export
class Kosha:
	'Kosha allows you to build a context for code generation based on your repo and environment.'
	def __init__(self, dir: Path = None, install_skill: bool = False, xdg_dir: Path = None):
		self.root, self.xdg = Path(dir or repo_root() or '.'), xdg_dir or xdg_data_home()
		if install_skill: mv_skill_md(dir=self.root, dry_run=False)
		self.cp, self.ce = self.root.joinpath('.kosha','code.db'), self.xdg.joinpath('kosha','env.db')
		for p in (self.cp, self.ce): p.parent.mkdir(parents=True, exist_ok=True)
		self.codedb, self.envdb = database(self.cp), database(self.ce)
		self.code_st,self.env_st = self.codedb.get_store(path=str,hash=True),self.envdb.get_store(package=str,hash=True)
		self.pkg_st, self.pkgs = self.envdb.get_store('pkg_store', hash=True), self.envdb.t.packages
		self.env_pd, self.code_rd = self.envdb.t.pkg_deps, self.codedb.t.repo_deps
		self.pkgs.create(name=str, version=str, summary=str, uploaded_at=float, pk=['name','version'],
			 defaults=dict(uploaded_at='CURRENT_TIMESTAMP'), if_not_exists=True)
		self.env_pd.create(from_pkg=str, to_pkg=str, n_modules=int, if_not_exists=True, pk=['from_pkg','to_pkg'])
		self.code_rd.create(from_module=str, to_pkg=str, n_files=int, if_not_exists=True, pk=['from_module','to_pkg'])

In [ ]:
#| export
def pkg_url(pkg: str) -> str | None:
    'Return best web URL for pkg from importlib.metadata: Source Code > Repository > Home-page > first Project-URL.'
    m, urls = meta(pkg), {}
    for e in (m.get_all('Project-URL') or []):
        label, _, u = e.partition(',')
        urls[label.strip().lower()] = u.strip()
    return (urls.get('source code') or urls.get('repository') or urls.get('source') or
            urls.get('github') or m.get('Home-page') or next(iter(urls.values()), None))


In [ ]:
#| export
def pkg_doc(pkg) -> dict:
	'Build pkg_store content dict: content=summary+readme, metadata=json.'
	m = meta(pkg)
	_get, _req_re, fn = lambda k: m.get(k,''), re.compile(r'^[\w-]+'), lambda rm: rm.group(0).lower().replace('-','_')
	reqs = ' '.join({fn(rm) for r in (m.get_all('Requires-Dist') or []) if (rm:=_req_re.match(r))})
	summary = _get('Summary')
	cont = summary+'\n'+(m.get_payload() or _get('Description') or '')[:2000].strip()
	return dict(content=cont, metadata=dict(name=pkg, version=v(pkg), keywords=_get('Keywords'),
	                                        requires=reqs, summary=summary, url=pkg_url(pkg)))


In [ ]:
pkg_doc('httpx')

{'content': 'The next generation HTTP client.\n<p align="center">\n  <a href="https://www.python-httpx.org/"><img width="350" height="208" src="https://raw.githubusercontent.com/encode/httpx/master/docs/img/butterfly.png" alt=\'HTTPX\'></a>\n</p>\n\n<p align="center"><strong>HTTPX</strong> <em>- A next-generation HTTP client for Python.</em></p>\n\n<p align="center">\n<a href="https://github.com/encode/httpx/actions">\n    <img src="https://github.com/encode/httpx/workflows/Test%20Suite/badge.svg" alt="Test Suite">\n</a>\n<a href="https://pypi.org/project/httpx/">\n    <img src="https://badge.fury.io/py/httpx.svg" alt="Package version">\n</a>\n</p>\n\nHTTPX is a fully featured HTTP client library for Python 3. It includes **an integrated command line client**, has support for both **HTTP/1.1 and HTTP/2**, and provides both **sync and async APIs**.\n\n---\n\nInstall HTTPX using pip:\n\n```shell\n$ pip install httpx\n```\n\nNow, let\'s get started:\n\n```pycon\n>>> import httpx\n>>> r = 

In [ ]:
u = pkg_url('httpx')
assert u and u.startswith('http'), f'expected a URL, got {u!r}'
print('pkg_url httpx:', u)
# packages without Project-URL fall back to Home-page
assert pkg_url('fastcore') is not None, 'fastcore should have a URL'
print('pkg_url fastcore:', pkg_url('fastcore'))


pkg_url httpx: https://github.com/encode/httpx
pkg_url fastcore: https://github.com/AnswerDotAI/fastcore/


In [ ]:
#| export
def has_init(d: Path) -> bool:
	'True if dir is a Python package root: has __init__.py or a C-extension __init__*.so.'
	return (d / '__init__.py').exists() or bool(list(d.glob('__init__*.so')))

In [ ]:
#| export
def enrich_chunks(content: L) -> L:
	'Set public_api+docstring flags on non-assign chunks and explode ClassDefs into method sub-chunks.'
	_m, _c = lambda d, k: d.get('metadata',{}).get(k,''), lambda d: d.get('content','')
	assigns, non_assigns = content.partition(lambda d: d['metadata'].get('type','') in ('Assign','AnnAssign'))
	chk = lambda e: isinstance(e, ast.Constant) and isinstance(e.value, str)
	def _get__all(d):
		try: return set(L(ast.parse(_c(d)).body[0].value.elts).filter(chk).attrgot('value'))
		except: return set()
	all_map = merge(*assigns.filter(lambda d: '__all__' in _c(d)).map(lambda d: {_m(d,'path'): _get__all(d)}))
	fwa = set(all_map)
	def _enrich(d):
		nm, pth = _m(d,'name'), _m(d,'path')
		try: doc = ast.get_docstring(ast.parse(_c(d)).body[0])
		except: doc = None
		return d | {'metadata': d['metadata'] | dict(docstring=doc,
			public_api=(nm in all_map[pth]) if pth in fwa else bool(nm and not nm.startswith('_')))}
	def _cls_methods(d):
		src, m = _c(d), d['metadata']
		try: cls = ast.parse(src).body[0]
		except: return L()
		if not isinstance(cls, ast.ClassDef): return L()
		pub, mod, off = m.get('public_api',False), m.get('mod_name',''), (m.get('lineno') or 1)-1
		is_fn = lambda n: isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)) and gs(src, n)
		mk = lambda n: d | {'content': gs(src, n).strip(), 'metadata': m | dict(
			name=n.name, type=type2str(n.__class__), mod_name=f'{mod}.{n.name}',
			public_api=pub and (not n.name.startswith('_') or n.name.startswith('__')),
			docstring=ast.get_docstring(n), lineno=off+n.lineno, end_lineno=off+(n.end_lineno or n.lineno))}
		return L(cls.body).filter(is_fn).map(mk)
	enriched = arun(parallel_async(_enrich, non_assigns)).concat()
	return enriched + enriched.filter(lambda d: d['metadata'].get('type')=='ClassDef').map(_cls_methods).concat()

In [ ]:
#| export
os.environ['TOKENIZERS_PARALLELISM']='false'  # to suppress warnings from tokenizers
@patch
def nuke(self:Kosha):
	'Reset the databases by dropping all tables.'
	for p in (self.cp, self.ce): p.parent.delete()

@patch
def _is_pkg_ingested(self:Kosha, pkg:str) -> bool:
	'Check if a package is already ingested and up-to-date.'
	return first(self.pkgs(select='name, version', where=f'name={pkg!r} and version={v(pkg)!r}'))

def embed_chunk(chunk:list|L,emb_fn=emb_doc, efn=embedder):
	'Embed a list of code chunks using emb_f'
	if not (chunk): return
	c = [b['content'] for b in chunk if b['content'] and b['content'].strip()]
	if not c: return
	for e,b in zip(emb_fn(efn)(c),chunk): b['embedding'] = e.tobytes()
	return chunk

def process_content(store, content:L, embed:bool=True, efn=embedder):
	'Embed chunks and upsert into store; no-op if content is empty.'
	if not content: return
	if embed: content = embed_chunk(content, efn=efn)
	return store.insert_all(content, upsert=True, hash_id='id', hash_id_columns=['content'])

@patch
@fdelegates(process_content)
def process_env(self:Kosha, content=None, reembed=False, **kwargs):
	'Embed all documents in the env store, or only those without embeddings if reembed=False.'
	content = content or self.env_st(where=f'embedding is NULL' if not reembed else None)
	return process_content(self.env_st, content, **kwargs)

def _slug(s): return hash_record({"content": s}) if s else None

def _content(store, where:str=None, f=None):
	'Helper to retrieve content from a store with optional filtering and transformation.'
	c=L(store(select='content', where=where)).filter(lambda c: c['content'] and c['content'].strip()).attrgot('content')
	return c if not f else L(c).map(f)

def count_imp(files, own:str='') -> Counter:
	'External top-level imports from a source-string iterable. Reuses cached _parse.'
	c, fn = Counter(), lambda p: p and p!=own and p[0]!='_'
	def do(f): c.update(set(L(parse(p=f)[1].values()).filter(fn)))
	L(files).map(do)
	return c

@patch
@fdelegates(pkg2chunks)
def update_pkg(self:Kosha, pkg:str, embed=True, exts=code_exts, efn=embedder, verbose=True, force=False, **kwargs):
    'Update package metadata in the packages table.'
    assert (o:= spec(pkg)), f'pkg {pkg} is not in environment'
    if verbose: print(f'updating pkg: {pkg} ...')
    ep = self._is_pkg_ingested(pkg)
    if ep and len(self.env_st(where=f'package={pkg!r}')) > 0:
        if verbose: print(f'package {ep} already loaded.')
        if not force: return
        print(f'forcing an update to pkg {ep}')
    pkg_par = Path(o.origin).parent.parent if o.origin else Path(o.submodule_search_locations[0])
    mod_fn = lambda p, n: '.'.join(list(Path(p).relative_to(pkg_par).with_suffix('').parts) + ([n] if n else []))
    _get = lambda o,k1=None: o.get('metadata').get(k1,'')
    meta_fn = lambda d: d['metadata'] | dict(mod_name=mod_fn(_get(d,'path'), _get(d,'name') or None),
                                             package=pkg, version=v(pkg))
    fn = lambda d: dict(package=pkg, uploaded_at=_get(d,'uploaded_at'), metadata=meta_fn(d))
    content = parallel(file_parse, pkg2files(pkg, exts=exts, **kwargs), threadpool=True, assigns=True,
                       imports=kwargs.pop('imports', False)).filter(true).concat().map(lambda d: d | fn(d))
    content, doc = enrich_chunks(content), pkg_doc(pkg)
    ex = L() if force else _content(self.env_st, where=f'package={pkg!r}', f=_slug)
    cont_hash = {_slug(d['content']): d for d in content}
    if del_ids:=set(ex).difference(cont_hash.keys()): self.env_st.delete_where(where='id in ("%s")'%'","'.join(del_ids))
    ch = filter_keys(cont_hash, not_(in_(ex))).values()
    if ch and self.process_env(ch, embed=embed, efn=efn):
        counts = count_imp(pkg2files(pkg, exts=['.py']),pkg.replace('-','_').split('.')[0])
        rows = [dict(from_pkg=pkg, to_pkg=dep, n_modules=n) for dep,n in counts.items() if spec(dep)]
        if rows: self.envdb.t.pkg_deps.insert_all(rows, replace=True)
    if verbose: print(f'updated pkg: {pkg} with {len(ch)} new/changed chunks, {len(ex)-len(ch)} unchanged, '
              f'{len(ex)-len(cont_hash)} removed')
    return self.pkgs.insert(dict(name=pkg, version=v(pkg), summary=_get(doc,'summary')), ignore=True)

@patch
def rm_pkg(self:Kosha, pkg:str, ver:str=None):
	'Remove a package and its code snippets from the database.'
	wh = f'package={pkg!r} AND version={ver!r}' if ver else f'package={pkg!r}'
	for t in [self.env_st, self.pkgs]: t.delete(where=wh)
	self.pkg_st.delete_where(where=f"json_extract(metadata,'$.name')={pkg!r}")
	self.env_pd.delete_where(where=f'from_pkg={pkg!r}')
	self.pkgs.delete_where(where=f'name={pkg!r}')

@patch
def process_repo(self:Kosha, content=None, reembed=False, **kwargs):
	'Embed all documents in the code store, or only those without embeddings if reembed=False.'
	content = content or self.code_st(where=f'embedding is NULL' if not reembed else None)
	return process_content(self.code_st, content, **kwargs)

@patch
def update_pkgs(self:Kosha,
    pkgs:str|list=None, # packages to sync; None syncs all installed env packages
    embed=True,         # embed chunks after indexing
    exts=code_exts,     # file extensions to include
    efn=embedder,       # embedding function
    verbose=True,       # print progress
    force=False,        # reindex even if package version already loaded
    **kwargs            # forwarded to update_pkg
):
	'Sync installed packages into the env store; if pkgs is None, syncs all env packages.'
	pkgs = set(pkgs).intersection(env_pkg_versions(pyproject=False)) if pkgs else set(env_pkg_versions())
	if verbose: print(f'loading pkgs {pkgs} ......')
	kw = dict(embed=embed, exts=exts, efn=efn, verbose=verbose, force=force, **kwargs)
	for pkg in tqdm(pkgs, desc='Updating packages', unit='pkg'): self.update_pkg(pkg, **kw)

@patch
@fdelegates(dir2files)
def update_repo(self:Kosha,
				dir:Path=None,    # directory to index; defaults to repo root
				embed:bool=True,  # embed chunks after parsing
				files:L=None,     # specific paths to (re)index; None = full sync
                exts:str=code_exts,
                efn=embedder,       # embedding function to use for code snippets
                verbose=True,       # verbose
                force=False,        # reindex all files regardless of mtime
				**kwargs              # extra args forwarded to dir2files
				):
	'Index or update repo code chunks. Pass files= for incremental update (e.g. from watcher).'
	dir = Path(dir or self.root)
	if files is None:
		known = {r['path']: r['uploaded_at'] for r in self.code_st(select='path, uploaded_at') if r['path']}
		all_files = dir2files(str(dir), strict_skip_file_re, strict_skip_folder_re,exts=exts, **kwargs)
		to_remove = L(known.keys()).filter(lambda k: k not in all_files.map(str)).concat()
		ch = all_files if force else L(all_files).filter(lambda f: str(f) not in known or known[str(f)] != f.stat().st_mtime)
	else: ch,to_remove = L(files).map(Path).partition(lambda f: f.exists() and f.suffix in exts)
	if to_remove: self.code_st.delete_where(where=f'path in ({",".join(to_remove.map(repr))})')
	if not ch: return
	if verbose: print(f'syncing files {ch} .....')
	o = Path(str(ch[0])).parent
	while has_init(o): o = o.parent
	mod_fn = lambda p, n: '.'.join(list(Path(p).relative_to(o).with_suffix('').parts)+([n] if n else []))
	_get = lambda m,k1,k2: m.get(k1,{}).get(k2,'')
	meta_fn = lambda d: d['metadata'] | dict(mod_name=mod_fn(_get(d,'metadata','path'),_get(d,'metadata','name')))
	fn = lambda d: dict(path=_get(d,'metadata','path'),uploaded_at=_get(d,'metadata','uploaded_at'),metadata=meta_fn(d))
	content = enrich_chunks(parallel(file_parse, ch, threadpool=True, progress=True, assigns=True).concat().map(
		lambda d: d | fn(d)))
	pl = ','.join(map(repr, L(ch).map(str)))
	ex = L() if force else _content(self.code_st, where=f'path IN ({pl})', f=_slug)
	cont_hash = {_slug(d['content']): d for d in content}
	if dids := set(ex).difference(cont_hash.keys()): self.code_st.delete_where(where='id in ("%s")' % '","'.join(dids))
	self.process_repo(filter_keys(cont_hash, not_(in_(ex))).values(), embed, efn=efn)
	for f in ch: self.codedb.q(f'update {self.code_st.name} set uploaded_at=? where path=?',[f.stat().st_mtime, str(f)])
	own = Path(dir).resolve().name
	rows = [dict(from_module=own, to_pkg=dep, n_files=n) for dep,n in count_imp(ch,own).items() if spec(dep)]
	if rows: self.code_rd.insert_all(rows, replace=True)
	if verbose: print('synced repo')

@patch
def prune_old_versions(self:Kosha, pkg:str):
	'Keep only the latest version of a package in the database.'
	if len(vers := self.pkgs(select='version', where=f'name={pkg!r}')) <= 1: return
	latest = sorted(L(vers).attrgot('version'), key=lambda v: tuple(map(int, v.split('.'))))[-1]
	je = lambda k, o='=', v='': f"json_extract(metadata, '$.{k}') {o} {v!r}"
	self.env_st.delete_where(where=f"{je('name')}={pkg!r} AND {je('version')}<>{latest!r}")
	self.pkgs.delete_where(where=f'name={pkg!r} AND version<>{latest!r}')

@patch
def prune_old_pkgs(self:Kosha):
	'Keep only the latest version of each package in the database.'
	L(self.pkgs(select='name')).attrgot('name').map(self.prune_old_versions)

@patch
def pkgs_in_env(self:Kosha, pyproject=False) -> list:
	'Intersection of the packages table with packages installed in the environment.'
	st_pkgs, inst_pkgs = L(self.pkgs(select='name, version')), env_pkg_versions(pyproject=pyproject)
	return st_pkgs.filter(lambda p: p['name'] in inst_pkgs and inst_pkgs[p['name']] == p['version'])

In [ ]:
import kosha.core as _kc, inspect

In [ ]:
_f = Path(inspect.getfile(_kc))
_anchor = _f.parent
while has_init(_anchor): _anchor = _anchor.parent
_mod = '.'.join(_f.relative_to(_anchor).with_suffix('').parts)
assert _mod == 'kosha.core', f'expected kosha.core, got {_mod}'
print('mod_name anchor fix: ok —', _mod)

mod_name anchor fix: ok — kosha.core


In [ ]:
dir2files(repo_root(), strict_skip_file_re, strict_skip_folder_re, exts=code_exts)

[Path('/Users/71293/code/personal/orgs/kosha/kosha/core.py'), Path('/Users/71293/code/personal/orgs/kosha/kosha/graph.py')]

In [ ]:
k = Kosha()

In [ ]:
pkg='fastcore'
k._is_pkg_ingested(pkg)

{'name': 'fastcore', 'version': '1.12.43'}

In [ ]:
k.env_st(select='content, embedding, json_extract(metadata, "$.mod_name","$.docstring", "$.public_api") as pa',where='package="httpx" and json_extract(metadata, "$.public_api")=1', limit=1)

[{'content': 'def request(\n    method: str,\n    url: URL | str,\n    *,\n    params: QueryParamTypes | None = None,\n    content: RequestContent | None = None,\n    data: RequestData | None = None,\n    files: RequestFiles | None = None,\n    json: typing.Any | None = None,\n    headers: HeaderTypes | None = None,\n    cookies: CookieTypes | None = None,\n    auth: AuthTypes | None = None,\n    proxy: ProxyTypes | None = None,\n    timeout: TimeoutTypes = DEFAULT_TIMEOUT_CONFIG,\n    follow_redirects: bool = False,\n    verify: ssl.SSLContext | str | bool = True,\n    trust_env: bool = True,\n) -> Response:\n    """\n    Sends an HTTP request.\n\n    **Parameters:**\n\n    * **method** - HTTP method for the new `Request` object: `GET`, `OPTIONS`,\n    `HEAD`, `POST`, `PUT`, `PATCH`, or `DELETE`.\n    * **url** - URL for the new `Request` object.\n    * **params** - *(optional)* Query parameters to include in the URL, as a\n    string, dictionary, or sequence of two-tuples.\n    * **c

In [ ]:
k.pkgs(order_by='name', limit=1)

[{'name': 'Jinja2',
  'version': '3.1.6',
  'summary': 'A very fast and expressive template engine.',
  'uploaded_at': '2026-04-24 02:27:22'}]

In [ ]:
k.update_pkg('protobuf')

updating pkg: protobuf ...
updated pkg: protobuf with 764 new/changed chunks, -764 unchanged, -764 removed


{'name': 'protobuf',
 'version': '7.34.1',
 'summary': '',
 'uploaded_at': '2026-04-29 10:58:07'}

In [ ]:
k.update_pkg('httpx',embed=False, force=True)

updating pkg: httpx ...
package {'name': 'httpx', 'version': '0.28.1'} already loaded.
forcing an update to pkg {'name': 'httpx', 'version': '0.28.1'}
updated pkg: httpx with 514 new/changed chunks, -514 unchanged, -514 removed


{}

In [ ]:
k.update_repo(skip_file_glob='*.ipynb', force=True)

syncing files [Path('/Users/71293/code/personal/orgs/kosha/kosha/core.py'), Path('/Users/71293/code/personal/orgs/kosha/kosha/graph.py')] .....
synced repo                                                          


In [ ]:
k.code_st(select='content, embedding, json_extract(metadata, "$.mod_name","$.docstring", "$.public_api") as pa',where='json_extract(metadata, "$.public_api")=1')

[{'content': "def process_content(store, content:L, embed:bool=True, efn=embedder, sz:int=500):\n\t'Embed chunks and upsert into store; no-op if content is empty.'\n\tif not content: return\n\tif embed: content = parallel(embed_chunk, L(chunked(content, sz)), efn=efn, progress=True, threadpool=True).concat()\n\treturn store.insert_all(content, upsert=True, hash_id='id', hash_id_columns=['content'])",
  'embedding': b']*b,\xa1\'* \xae\x10\xa2\xa3\xc1&\x88)\xfd \xfa"\xcb\xa9\x08\xab\x03\xa8\r"\n\xaco$\x00\x1e\x19(E\xa9?)]\x1c\xa9\'"*\xbb\x9c\xda\x9a7%\x87(\x98\x9f\xdc\xa64\x1cc\x8b\xf4\xa2\x95\xa2\xbc\x1a\x84\xa9\x1c N\x1c<\xa4\xd5\xaa2\xa5\xa1\x1c@\x9c\xef(\x94\xab6\xabk(\x84\x8d\xf5#;(#\xa1c\xa4p\x9e\xc6\xaa\x17\xa8_+\xe8\x1c`\x1f\x8e+\x83!`\x9fr!\xe4\xa9\x8d\xa0\x1c\xa9\xf6\xa8\xc6,>\xa5#(\xa2\'y\'c\x1d>\xa55\x1ex\xa9\xd4\x1a\x0f\x9a8\xad\x9a(\xb0&\x9a\xa8\xcd+q\x96\x02$\xd6\xab\x1a\xae\x86\x1ep\xa8\x16(3\xa8\xfb*0\x15\xc2&d\xa30\x91\x8d\xae\xc5\xa6\xe3\x9f\x11\xaa^\xa8\xbc\xa8o\xa9$\

In [ ]:
k.update_pkgs(set(env_pkg_versions()), embed=False, force=True)

loading pkgs {'model2vec', 'pdf_oxide', 'onnxruntime', 'usearch', 'pillow', 'fastprogress', 'tokenizers', 'chonkie', 'watchfiles', 'python-fasthtml', 'pyan3', 'pandas', 'fastlite', 'codesigs', 'notebook', 'anyio', 'fastcore', 'yake', 'litesearch'} ......


Updating packages:   0%|          | 0/19 [00:00<?, ?pkg/s]

updating pkg: model2vec ...
package {'name': 'model2vec', 'version': '0.8.1'} already loaded.
forcing an update to pkg {'name': 'model2vec', 'version': '0.8.1'}


Updating packages:   5%|▌         | 1/19 [00:00<00:06,  2.72pkg/s]

updated pkg: model2vec with 112 new/changed chunks, -112 unchanged, -112 removed
updating pkg: pdf_oxide ...
package {'name': 'pdf_oxide', 'version': '0.3.39'} already loaded.
forcing an update to pkg {'name': 'pdf_oxide', 'version': '0.3.39'}
updated pkg: pdf_oxide with 21 new/changed chunks, -21 unchanged, -21 removed
updating pkg: onnxruntime ...
package {'name': 'onnxruntime', 'version': '1.25.1'} already loaded.
forcing an update to pkg {'name': 'onnxruntime', 'version': '1.25.1'}


...

updated pkg: fastcore with 890 new/changed chunks, -890 unchanged, -890 removed
updating pkg: yake ...
package {'name': 'yake', 'version': '0.7.3'} already loaded.
forcing an update to pkg {'name': 'yake', 'version': '0.7.3'}


Updating packages:  95%|█████████▍| 18/19 [00:58<00:03,  3.59s/pkg]

updated pkg: yake with 129 new/changed chunks, -129 unchanged, -129 removed
updating pkg: litesearch ...
package {'name': 'litesearch', 'version': '0.0.23'} already loaded.
forcing an update to pkg {'name': 'litesearch', 'version': '0.0.23'}


Updating packages: 100%|██████████| 19/19 [00:59<00:00,  3.11s/pkg]

updated pkg: litesearch with 62 new/changed chunks, -62 unchanged, -62 removed


In [ ]:
k.env_st(where='embedding is NULL', select='count(*)')

[{'count(*)': 0}]

In [ ]:
k.update_pkg('apswutils',force=True, embed=False)

updating pkg: apswutils ...
package {'name': 'apswutils', 'version': '0.1.2'} already loaded.
forcing an update to pkg {'name': 'apswutils', 'version': '0.1.2'}
updated pkg: apswutils with 185 new/changed chunks, -185 unchanged, -185 removed


{}

In [ ]:
k.process_env()

<Table store (id, content, embedding, metadata, uploaded_at, package)>

In [ ]:
k.update_pkgs(['python-fasthtml','fastcore','litesearch','fastlite','apsw','ghapi','httpx'], embed=True)

loading pkgs {'httpx', 'ghapi', 'fastcore', 'fastlite', 'python-fasthtml', 'apsw', 'litesearch'} ......


Updating packages:  29%|██▊       | 2/7 [00:00<00:00, 17.95pkg/s]

updating pkg: httpx ...
package {'name': 'httpx', 'version': '0.28.1'} already loaded.
updating pkg: ghapi ...
package {'name': 'ghapi', 'version': '1.0.13'} already loaded.
updating pkg: fastcore ...
package {'name': 'fastcore', 'version': '1.12.43'} already loaded.
updating pkg: fastlite ...


Updating packages: 100%|██████████| 7/7 [00:00<00:00, 19.05pkg/s]

package {'name': 'fastlite', 'version': '0.2.4'} already loaded.
updating pkg: python-fasthtml ...
package {'name': 'python-fasthtml', 'version': '0.13.4'} already loaded.
updating pkg: apsw ...
package {'name': 'apsw', 'version': '3.53.0.0'} already loaded.
updating pkg: litesearch ...
package {'name': 'litesearch', 'version': '0.0.23'} already loaded.


## Structured Search
> `parseq` strips `key:value` filter tokens from a query string. `filt2wh` maps them to SQL WHERE clauses. `Kosha.context` fans out searches across detected packages and merges results with chained RRF.

In [ ]:
#| export
_filter_keys = frozenset({'file', 'files', 'path', 'paths', 'package', 'packages', 'lang', 'langs', 'type', 'types'})
_filter_pat = re.compile(r'\b(' + '|'.join(_filter_keys) + r'):(\S+)')
_filter_norm = {'files': 'file', 'paths': 'path', 'packages': 'package', 'langs': 'lang', 'types': 'type'}

def parseq(q: str) -> tuple:
    'Parse key:value filter tokens from a query. Returns (bare_query, filters_dict).'
    filters = defaultdict(list)
    for m in _filter_pat.finditer(q):
        key = _filter_norm.get(m.group(1), m.group(1))
        filters[key].extend(m.group(2).split(','))
    return _filter_pat.sub('', q).strip() or q, filters

In [ ]:
parseq('how do I merge dicts package:fastcore path:data*')

('how do I merge dicts',
 defaultdict(list, {'package': ['fastcore'], 'path': ['data*']}))

In [ ]:
assert parseq('stripe payments package:fasthtml file:routes*') == ('stripe payments', {'package': ['fasthtml'], 'file': ['routes*']})
assert parseq('package:monsterui package:fasthtml create a table') == ('create a table', {'package': ['monsterui', 'fasthtml']})
assert parseq('no filters here') == ('no filters here', {})
assert parseq('type:FunctionDef lang:py') == ('type:FunctionDef lang:py', {'type': ['FunctionDef'], 'lang': ['py']})
assert parseq('package:only')[0] == 'package:only'
# plural forms + comma-separated values
assert parseq('merge dicts packages:fastcore,litesearch') == ('merge dicts', {'package': ['fastcore', 'litesearch']})
assert parseq('search paths:basics,core,data') == ('search', {'path': ['basics', 'core', 'data']})
assert parseq('query packages:fastcore,litesearch paths:basics,core') == ('query', {'package': ['fastcore', 'litesearch'], 'path': ['basics', 'core']})
assert parseq('langs:py,js files:routes*,api*') == ('langs:py,js files:routes*,api*', {'lang': ['py', 'js'], 'file': ['routes*', 'api*']})
print('parse_query: all assertions pass')

parse_query: all assertions pass


In [ ]:
#| export
_glob2like = {'*': '%', '?': '_'}
def _glob_to_like(pat: str) -> str:
    'Convert a glob pattern (*, ?) to a SQL LIKE pattern with surrounding % anchors.'
    p = pat.replace('*', '%').replace('?', '_')
    if not p.startswith('%'): p = '%' + p
    if not p.endswith('%'): p += '%'
    return p

_ext = {'py': '.py', 'js': '.js', 'ts': '.ts', 'jsx': '.jsx', 'tsx': '.tsx', 'rb':'.rb','rs':'.rs','go':'.go',
        'java':'.java','cs':'.cs', 'swift':'.swift','kt':'.kt','lua':'.lua','php':'.php','python':'.py',
        'javascript':'.js','typescript':'.ts','ruby':'.rb','rust':'.rs','golang':'.go'}

def filt2wh(filters: dict, store: str = 'code') -> str | None:
	'Convert a filter dict (from parse_query) to a SQL WHERE clause for the given store (code or env).'
	_get, je = lambda k: filters.get(k,[]), lambda k, o='=', v='': f"json_extract(metadata, '$.{k}') {o} {v!r}"
	list_or = lambda it: ['(%s)'%' OR '.join(ls)] if (ls:=listify(it)) else []
	c, files, paths, langs = [], _get('file'), _get('path'), _get('lang')
	path_wh, lang_chk = list(map(_glob_to_like, files + paths)), list(map(lambda l: _ext.get(l, f'.{l}'), langs))
	lang_wh = list_or(map(lambda lc: je('lang',v=lc), lang_chk))
	if store == 'code': c = list_or(map(lambda p: f'path LIKE {p!r}',path_wh)) + lang_wh
	elif store == 'env':
		c = list_or(map(lambda p: je('path','like',p),path_wh)) + lang_wh
		if pkgs:=_get('package'): c += list_or(map(lambda p: f'package={p!r}',pkgs))
	c += list_or(map(lambda r: je('type',v=r), _get('type')))
	return ' AND '.join(c) if c else None

In [ ]:
filt2wh(parseq('how do I merge dicts package:fastcore path:data*')[1], 'env')

"(json_extract(metadata, '$.path') like '%data%') AND (package='fastcore')"

In [ ]:
filt2wh({'type': ['FunctionDef']}, 'code')

"(json_extract(metadata, '$.type') = 'FunctionDef')"

In [ ]:
assert filt2wh({'file': ['routes*']}, 'code') == "(path LIKE '%routes%')"
assert filt2wh({'path': ['api/*']}, 'code') == "(path LIKE '%api/%')"
assert filt2wh({'lang': ['py']}, 'code') == "(json_extract(metadata, '$.lang') = '.py')"
assert filt2wh({'lang': ['python']}, 'code') == "(json_extract(metadata, '$.lang') = '.py')"
assert filt2wh({'package': ['fasthtml']}, 'env') == "(package='fasthtml')"
assert filt2wh({'package': ['fasthtml', 'monsterui']}, 'env') == "(package='fasthtml' OR package='monsterui')"
assert filt2wh({'type': ['FunctionDef']}, 'code') == "(json_extract(metadata, '$.type') = 'FunctionDef')"
assert filt2wh({}, 'code') is None
assert filt2wh({}, 'env') is None
assert filt2wh({'package': ['fasthtml']}, 'code') is None
print('filters_to_where: all assertions pass')

filters_to_where: all assertions pass


In [ ]:
#| export
@patch
def env_context(self:Kosha,
				q:str,               	# query to search (supports key:value filters)
				emb_q:str=None,     	# query to embed; defaults to bare query after filter parsing
				wide:bool=False,    	# whether to use wide search
				emb_fn=emb_query,		# embedding function
                columns:str='content,metadata,package',			# comma separated columns string to return from search
                where:str=None,			# additional where clause to filter search results
                efn=embedder,			# embedding function to use for code snippets when updating packages
				sys_wide=True,
				**kw					# additional args to pass to db.search
				) -> L:
	'Code search through the database to find relevant code snippets.'
	raw, fs = parseq(q)
	emb= emb_fn(efn)(emb_q or raw)
	all_p = set(raw.split()).intersection(self.pkgs2consider(sys_wide)) if 'package' not in fs else []
	for p in all_p: fs['package'].append(p)
	wh = ' AND '.join(map(lambda p: f'({p})', L(filt2wh(fs, 'env'), where).filter(true)))
	fn = lambda r: r | dict(metadata=jl(r['metadata'])) if 'metadata' in r else r
	return L(self.envdb.search(pre(raw, wide=wide), emb.tobytes(), columns.split(','), where=wh, **kw)).map(fn)

@patch
def pkgs2consider(self: Kosha, sys_wide=True) -> set:
	'Determine which packages to consider for search based on the intersection of the packages table and the environment.'
	ex_pkgs = merge(*L(self.pkgs(select='name, version')).map(lambda d: {d['name']: d['version']}))
	if sys_wide: return ex_pkgs.keys()
	env_pkgs = env_pkg_versions()
	return {p for p in env_pkgs if p in ex_pkgs and env_pkgs[p] == ex_pkgs[p]}

In [ ]:
#| export
@patch
def repo_context(self:Kosha,
                q:str,                          # query to search (supports key:value filters)
                emb_q:str=None,                 # query to embed; defaults to bare query after filter parsing
                wide:bool=False,                # use wide (OR) FTS search
                emb_fn=emb_query,               # embedding function
                columns:str='content,path,metadata', # columns to return
                where:str=None,                 # extra SQL filter
                efn=embedder,                   # embedding function to use for code snippets when updating repo
                **kw                            # additional args to pass to db.search
) -> L:
	'Semantic + keyword search through indexed repo code.'
	raw, fs = parseq(q)
	emb = emb_fn(efn)(emb_q or raw)
	wh = filt2wh(fs, 'code')
	if where and wh: wh = f'({where}) AND ({wh})'
	elif where: wh = where
	fn = lambda r: r | dict(metadata=jl(r['metadata'])) if 'metadata' in r else r
	return L(self.codedb.search(pre(raw, wide=wide), emb.tobytes(), columns.split(','), where=wh, **kw)).map(fn)

@patch
async def awatch_repo(self:Kosha, dir:Path=None, **kw):
	'Async watcher: re-indexes changed files on every watchfiles event.'
	from watchfiles import awatch
	dir = Path(dir or self.root)
	async for changes in awatch(str(dir)): self.update_repo(dir, files=L({c[1] for c in changes}), **kw)

@patch
def watch_repo(self:Kosha, dir:Path=None, **kw):
	'Block and watch repo for changes, re-indexing incrementally. Ctrl-C to stop.'
	arun(self.awatch_repo(dir, **kw))

In [ ]:
#| export
@patch
def pkg_context(self:Kosha,
                q:str, # the search query
                limit:int=10, # limits
                efn=embedder
                ) -> L:
	'FTS5+vector search over package descriptions in pkg_store.'
	emb = emb_query(efn)(q).tobytes()
	fn = lambda r: r | dict(metadata=jl(r['metadata'])) if isinstance(r.get('metadata'), str) else r
	return L(self.envdb.search(q, emb, columns=['content','metadata'], table_name='pkg_store', limit=limit)).map(fn)

In [ ]:
first(k.repo_context('how do I parse python ast', limit=1))

{'rowid': 138,
 'content': 'def parse(code=None, p=None):\n    "Parse source, tag parents, return (tree, imp, top_fns, all_fns). Cached — called freely."\n    if not code and not p: return None, {}, set(), set()\n    try: tree = ast.parse(Path(p).read_text(errors=\'replace\') if not code else code)\n    except SyntaxError: return None, {}, set(), set()\n    [setattr(c,\'parent\',n) for n in ast.walk(tree) for c in ast.iter_child_nodes(n)]\n    is_top = lambda n: isinstance(getattr(n,\'parent\',None), ast.Module)\n    is_fn  = lambda n: isinstance(n,(ast.FunctionDef,ast.AsyncFunctionDef,ast.ClassDef))\n    is_nm  = lambda n: isinstance(n, ast.Name)\n    imp = {}\n    for n in ast.walk(tree):\n        if isinstance(n, ast.Import):\n            for a in n.names: imp[a.asname or a.name.split(\'.\')[0]] = a.name.split(\'.\')[0]\n        elif isinstance(n, ast.ImportFrom):\n            pkg = (n.module or \'\').split(\'.\')[0]\n            for a in n.names:\n                if a.name != \'*\'

In [ ]:
k.env_context('payment page with fasthtml', limit=10).map(lambda d: d['metadata']['package'])

['python-fasthtml', 'jupyterlab', 'python-fasthtml', 'notebook', 'python-fasthtml', 'python-fasthtml', 'python-fasthtml', 'fastcore', 'notebook', 'notebook']

In [ ]:
k.env_context('payment page with fasthtml package:fastcore', limit=10).map(lambda d: d['metadata']['package'])

['fastcore', 'fastcore', 'fastcore', 'fastcore', 'fastcore', 'fastcore', 'fastcore', 'fastcore', 'fastcore', 'fastcore']

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastprogress/fastprogress.py:171: UserWarning: Couldn't import ipython display functions, progress bar will use console behavior
  warn("Couldn't import ipython display functions, progress bar will use console behavior")
